[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinculum3141-ship-it/bike-availability-data-science/blob/main/notebooks/Module_02_Data_Acquisition/M2_02_weather_data_api.ipynb)

# 🌦️ Module 02: Weather Data Acquisition

**Purpose**: Learn to fetch historical weather data from the Open-Meteo API  
**Module**: Module 02 - Data Acquisition  
**Author**: [Your Name]  
**Date**: 2026-01-15

---

## 📋 Overview

In this notebook, you will:
- [ ] Understand the Open-Meteo API structure
- [ ] Fetch historical weather data for Amsterdam
- [ ] Parse weather variables (temperature, precipitation, wind, etc.)
- [ ] Handle time zones and timestamps
- [ ] Aggregate hourly weather data
- [ ] Save processed weather data
- [ ] Match weather data to bike data timestamps

**Goal**: Build confidence working with weather APIs and time series data!

**Estimated Time**: 90-120 minutes

---

## 🎓 Learning Path Note

**This Module**: Focus on exploring weather APIs and understanding data structures through notebook experiments.

**Module 8 - Automation**: You'll learn to convert these exploration patterns into production-ready scripts.

**Learning Strategy**: 
- ✅ DO: Experiment with API calls and data transformations here
- ⏸️ LATER: Automate data pipelines using reusable scripts (Module 8)

---

---

## 📖 Part 1: Understanding Weather APIs

### What is Open-Meteo?

[Open-Meteo](https://open-meteo.com/) is a **free, open-source weather API** with:
- ✅ **No API key required** - immediate access
- ✅ **Historical data** - from 1940 to present
- ✅ **Forecast data** - 16-day forecasts
- ✅ **High resolution** - hourly and daily data
- ✅ **Global coverage** - worldwide locations
- ✅ **Fast and reliable** - cached data for performance

### Why Weather Data Matters for Bike Sharing

Weather significantly impacts bike usage:
- 🌧️ **Rain** → Fewer riders (people avoid getting wet)
- ❄️ **Cold** → Decreased usage (uncomfortable cycling)
- ☀️ **Sunny** → More riders (pleasant conditions)
- 💨 **Wind** → Can deter cycling (especially headwinds)
- 🌡️ **Temperature** → Optimal range ~15-25°C

### API Endpoints

**Base URL**: `https://archive-api.open-meteo.com/v1/archive`

**Required Parameters**:
- `latitude` - Location latitude
- `longitude` - Location longitude  
- `start_date` - Start date (YYYY-MM-DD)
- `end_date` - End date (YYYY-MM-DD)
- `hourly` - Weather variables to fetch

**Example Request**:
```
https://archive-api.open-meteo.com/v1/archive?
  latitude=52.37&
  longitude=4.90&
  start_date=2024-01-01&
  end_date=2024-01-31&
  hourly=temperature_2m,precipitation,windspeed_10m
```

### Available Weather Variables

| Variable | Description | Unit |
|----------|-------------|------|
| `temperature_2m` | Temperature at 2m height | °C |
| `relativehumidity_2m` | Relative humidity | % |
| `precipitation` | Total precipitation | mm |
| `rain` | Rain volume | mm |
| `snowfall` | Snowfall amount | cm |
| `windspeed_10m` | Wind speed at 10m | km/h |
| `winddirection_10m` | Wind direction | ° |
| `cloudcover` | Cloud cover | % |
| `pressure_msl` | Sea level pressure | hPa |

---

## 🔧 Part 2: Setup

Run this cell first to set up the environment.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 1. IMPORTS
# ═══════════════════════════════════════════════════════════

# Standard library
import os
import sys
import json
from datetime import datetime, timedelta
from pathlib import Path

# Third-party imports
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.dates import DateFormatter

# Configure pandas display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Check if running in Google Colab
if 'google.colab' in sys.modules:
    print("📍 Running in Google Colab")
else:
    print("📍 Running locally")

# Add project root to path
project_root = os.path.abspath('../..' if 'notebooks' in os.getcwd() else '.')
if project_root not in sys.path:
    sys.path.append(project_root)

print("✅ Setup complete!")
print(f"📁 Working directory: {os.getcwd()}")
print(f"🐍 Python version: {sys.version.split()[0]}")
print(f"📅 Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## ⚙️ Part 3: Configuration

Define API endpoints, location coordinates, and date ranges.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2. CONFIGURATION
# ═══════════════════════════════════════════════════════════

# API Configuration
BASE_URL = "https://archive-api.open-meteo.com/v1/archive"

# Amsterdam coordinates (city center)
AMSTERDAM_LAT = 52.3676
AMSTERDAM_LON = 4.9041

# Date range - last 30 days for demonstration
# (In practice, match this to your bike data dates)
END_DATE = datetime.now().date()
START_DATE = END_DATE - timedelta(days=30)

# Weather variables to fetch
WEATHER_VARS = [
    'temperature_2m',           # Temperature at 2 meters
    'relativehumidity_2m',      # Relative humidity
    'precipitation',            # Total precipitation
    'rain',                     # Rain volume
    'windspeed_10m',           # Wind speed at 10 meters
    'winddirection_10m',       # Wind direction
    'cloudcover',              # Cloud cover percentage
    'pressure_msl'             # Mean sea level pressure
]

# File paths
DATA_DIR = Path('../../data/raw') if 'notebooks' in os.getcwd() else Path('data/raw')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Output filename with timestamp
TIMESTAMP = datetime.now().strftime('%Y-%m-%d_%H%M%S')
OUTPUT_FILE = DATA_DIR / f'weather_amsterdam_{TIMESTAMP}.csv'

print("✅ Configuration set")
print(f"📍 Location: Amsterdam ({AMSTERDAM_LAT}, {AMSTERDAM_LON})")
print(f"📅 Date range: {START_DATE} to {END_DATE}")
print(f"🌦️ Weather variables: {len(WEATHER_VARS)}")
print(f"💾 Output file: {OUTPUT_FILE}")

---

## 🌐 Part 4: Fetch Weather Data

### Step 1: Build API Request URL

In [ ]:
# ═══════════════════════════════════════════════════════════
# 3. BUILD API URL
# ═══════════════════════════════════════════════════════════

# Build query parameters
params = {
    'latitude': AMSTERDAM_LAT,
    'longitude': AMSTERDAM_LON,
    'start_date': START_DATE.strftime('%Y-%m-%d'),
    'end_date': END_DATE.strftime('%Y-%m-%d'),
    'hourly': ','.join(WEATHER_VARS),
    'timezone': 'Europe/Amsterdam'  # Important for correct timestamps!
}

print("🔗 API Request Parameters:")
print("=" * 60)
for key, value in params.items():
    print(f"  {key:20s}: {value}")
print("=" * 60)

# The requests library will URL-encode these parameters
print(f"\n🌐 Base URL: {BASE_URL}")
print(f"📋 Full request will include {len(params)} parameters")

### 🧠 Your Task: Fetch Weather Data

**What You've Learned in Module 1:**
- In [M1_03_open_data_sources.ipynb](../Module_01_Introduction/M1_03_open_data_sources.ipynb), you saw how to:
  - Make API requests with query parameters
  - Use `requests.get(url, params=params_dict)`
  - Handle timeouts
  - Check for errors with try/except

**Now It's Your Turn:**
Complete the function below to fetch weather data from Open-Meteo API.

**Steps:**
1. Make a GET request with the params dictionary
2. Check the status code
3. Parse JSON and return the data
4. Handle errors appropriately

In [ ]:
# ═══════════════════════════════════════════════════════════
# 4. FETCH WEATHER DATA - YOUR TASK
# ═══════════════════════════════════════════════════════════

def fetch_weather_data(latitude, longitude, start_date, end_date, 
                       variables, timezone='Europe/Amsterdam', timeout=30):
    """
    Fetch historical weather data from Open-Meteo API.
    
    Parameters:
    -----------
    latitude : float
        Latitude coordinate
    longitude : float
        Longitude coordinate
    start_date : str or date
        Start date (YYYY-MM-DD)
    end_date : str or date
        End date (YYYY-MM-DD)
    variables : list
        List of weather variable names to fetch
    timezone : str
        Timezone for timestamps (default: 'Europe/Amsterdam')
    timeout : int
        Request timeout in seconds
    
    Returns:
    --------
    dict or None
        JSON data if successful, None otherwise
    """
    # Build query parameters
    params = {
        'latitude': latitude,
        'longitude': longitude,
        'start_date': start_date if isinstance(start_date, str) else start_date.strftime('%Y-%m-%d'),
        'end_date': end_date if isinstance(end_date, str) else end_date.strftime('%Y-%m-%d'),
        'hourly': ','.join(variables),
        'timezone': timezone
    }
    
    # TODO: Make the API request with error handling
    # Hint: Use requests.get(BASE_URL, params=params, timeout=timeout)
    # Hint: Check response.status_code
    # Hint: Use try/except for error handling (like you saw in M1_03)
    
    try:
        # Your code here...
        response = None  # Replace with actual request
        
        # TODO: Check status and return JSON
        # Hint: if response.status_code == 200: return response.json()
        
        return None  # Replace with your code
        
    except requests.exceptions.Timeout:
        print(f"⏱️ Timeout: Request took longer than {timeout} seconds")
        return None
        
    except requests.exceptions.RequestException as e:
        print(f"❌ Error fetching weather data: {e}")
        return None


# Test your function
weather_data = fetch_weather_data(
    AMSTERDAM_LAT, 
    AMSTERDAM_LON,
    START_DATE,
    END_DATE,
    WEATHER_VARS
)

### Step 3: Explore the JSON Response

After successfully fetching the data, explore its structure.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5. EXPLORE JSON STRUCTURE
# ═══════════════════════════════════════════════════════════

if weather_data:
    print("📊 Response Structure:")
    print("=" * 60)
    print(f"Top-level keys: {list(weather_data.keys())}")
    
    print("\n📍 Location Information:")
    print(f"  Latitude: {weather_data.get('latitude', 'N/A')}")
    print(f"  Longitude: {weather_data.get('longitude', 'N/A')}")
    print(f"  Elevation: {weather_data.get('elevation', 'N/A')} m")
    print(f"  Timezone: {weather_data.get('timezone', 'N/A')}")
    
    print("\n🌦️ Available hourly variables:")
    for var in weather_data['hourly'].keys():
        if var != 'time':
            sample_values = weather_data['hourly'][var][:3]
            print(f"  • {var:25s}: {sample_values}")
    
    print("\n⏰ Time range:")
    times = weather_data['hourly']['time']
    print(f"  First: {times[0]}")
    print(f"  Last:  {times[-1]}")
    print(f"  Total records: {len(times)}")

---

## 📊 Part 5: Convert to DataFrame

### 🧠 Your Task: Convert Weather JSON to DataFrame

**What You've Learned:**
- In Module 1, you converted bike data JSON to DataFrame
- In M2_01 above, you just practiced this again with bike stations
- The pattern is the same: extract the data, create DataFrame, parse timestamps

**Now It's Your Turn:**
Convert the weather JSON to a DataFrame.

**Steps:**
1. Extract the hourly data from `weather_data['hourly']`
2. Create a DataFrame from this dictionary
3. Parse the 'time' column to datetime
4. Display the result

In [ ]:
# ═══════════════════════════════════════════════════════════
# 6. CONVERT TO DATAFRAME - YOUR TASK
# ═══════════════════════════════════════════════════════════

# TODO: Extract hourly data from the JSON
# Hint: hourly_data = weather_data['hourly']
hourly_data = None  # Replace with your code

# TODO: Convert to DataFrame
# Hint: pd.DataFrame(dictionary_data)
df_weather = None  # Replace with your code

# TODO: Parse timestamps
# Hint: df_weather['time'] = pd.to_datetime(df_weather['time'])
# Your code here...

# TODO: Display the result
# Hint: Use .head(), .info(), .shape
# Your code here...

---

## 🔍 Part 6: Data Exploration

Let's explore the weather patterns!

In [ ]:
# ═══════════════════════════════════════════════════════════
# 7. SUMMARY STATISTICS
# ═══════════════════════════════════════════════════════════

print("📈 Weather Summary Statistics:")
print("=" * 60)

# Select only numeric columns for statistics
numeric_cols = df_weather.select_dtypes(include=[np.number]).columns.tolist()
# Exclude metadata columns
exclude_cols = ['latitude', 'longitude', 'elevation_m']
stat_cols = [col for col in numeric_cols if col not in exclude_cols]

display(df_weather[stat_cols].describe())

print("\n" + "=" * 60)
print("🌡️ Temperature Analysis:")
print("=" * 60)
print(f"Average Temperature: {df_weather['temperature_2m'].mean():.1f}°C")
print(f"Min Temperature: {df_weather['temperature_2m'].min():.1f}°C")
print(f"Max Temperature: {df_weather['temperature_2m'].max():.1f}°C")
print(f"Temperature Range: {df_weather['temperature_2m'].max() - df_weather['temperature_2m'].min():.1f}°C")

print("\n" + "=" * 60)
print("🌧️ Precipitation Analysis:")
print("=" * 60)
total_precip = df_weather['precipitation'].sum()
rainy_hours = (df_weather['precipitation'] > 0).sum()
rainy_pct = rainy_hours / len(df_weather) * 100
print(f"Total Precipitation: {total_precip:.1f} mm")
print(f"Hours with Rain: {rainy_hours} ({rainy_pct:.1f}%)")
print(f"Average Rain (when raining): {df_weather[df_weather['precipitation'] > 0]['precipitation'].mean():.2f} mm/hour")

print("\n" + "=" * 60)
print("💨 Wind Analysis:")
print("=" * 60)
print(f"Average Wind Speed: {df_weather['windspeed_10m'].mean():.1f} km/h")
print(f"Max Wind Speed: {df_weather['windspeed_10m'].max():.1f} km/h")
print(f"Calm hours (< 5 km/h): {(df_weather['windspeed_10m'] < 5).sum()}")
print(f"Windy hours (> 20 km/h): {(df_weather['windspeed_10m'] > 20).sum()}")

### Visualizations

In [ ]:
# ═══════════════════════════════════════════════════════════
# 8. WEATHER VISUALIZATIONS
# ═══════════════════════════════════════════════════════════

fig, axes = plt.subplots(3, 2, figsize=(16, 12))
fig.suptitle('🌦️ Amsterdam Weather Overview (Last 30 Days)', fontsize=16, fontweight='bold')

# 1. Temperature over time
axes[0, 0].plot(df_weather['timestamp'], df_weather['temperature_2m'], 
                color='coral', linewidth=1.5, alpha=0.8)
axes[0, 0].axhline(df_weather['temperature_2m'].mean(), color='red', 
                   linestyle='--', label=f"Mean: {df_weather['temperature_2m'].mean():.1f}°C")
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Temperature (°C)')
axes[0, 0].set_title('Temperature Over Time')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Temperature distribution
axes[0, 1].hist(df_weather['temperature_2m'], bins=30, color='coral', 
                edgecolor='black', alpha=0.7)
axes[0, 1].axvline(df_weather['temperature_2m'].mean(), color='red', 
                   linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Temperature (°C)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Temperature Distribution')

# 3. Precipitation over time
axes[1, 0].bar(df_weather['timestamp'], df_weather['precipitation'], 
               color='steelblue', alpha=0.7, width=0.04)
axes[1, 0].set_xlabel('Date')
axes[1, 0].set_ylabel('Precipitation (mm)')
axes[1, 0].set_title('Precipitation Over Time')
axes[1, 0].grid(True, alpha=0.3)

# 4. Wind speed over time
axes[1, 1].plot(df_weather['timestamp'], df_weather['windspeed_10m'], 
                color='green', linewidth=1, alpha=0.7)
axes[1, 1].axhline(df_weather['windspeed_10m'].mean(), color='darkgreen', 
                   linestyle='--', label=f"Mean: {df_weather['windspeed_10m'].mean():.1f} km/h")
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('Wind Speed (km/h)')
axes[1, 1].set_title('Wind Speed Over Time')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# 5. Humidity and cloud cover
axes[2, 0].plot(df_weather['timestamp'], df_weather['relativehumidity_2m'], 
                color='purple', linewidth=1, alpha=0.7, label='Humidity')
axes[2, 0].plot(df_weather['timestamp'], df_weather['cloudcover'], 
                color='gray', linewidth=1, alpha=0.7, label='Cloud Cover')
axes[2, 0].set_xlabel('Date')
axes[2, 0].set_ylabel('Percentage (%)')
axes[2, 0].set_title('Humidity and Cloud Cover')
axes[2, 0].legend()
axes[2, 0].grid(True, alpha=0.3)

# 6. Weather conditions summary (daily aggregation)
daily_weather = df_weather.set_index('timestamp').resample('D').agg({
    'temperature_2m': 'mean',
    'precipitation': 'sum',
    'windspeed_10m': 'mean'
})

x = np.arange(len(daily_weather))
width = 0.25

axes[2, 1].bar(x - width, daily_weather['temperature_2m'], width, 
               label='Avg Temp (°C)', color='coral', alpha=0.7)
axes[2, 1].bar(x, daily_weather['precipitation'], width, 
               label='Precip (mm)', color='steelblue', alpha=0.7)
axes[2, 1].bar(x + width, daily_weather['windspeed_10m'], width, 
               label='Avg Wind (km/h)', color='green', alpha=0.7)

axes[2, 1].set_xlabel('Days')
axes[2, 1].set_ylabel('Value')
axes[2, 1].set_title('Daily Weather Summary')
axes[2, 1].legend()
axes[2, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Visualizations created")

### 🧠 Task 6.1: Analyze Weather Impact on Biking

**Your Task**: Create a "bikeability score" based on weather conditions.

Consider these factors:
- Ideal temperature: 15-25°C (score = 1.0)
- Too cold (< 5°C) or too hot (> 30°C): score = 0.3
- Rain: reduce score proportionally
- High wind (> 25 km/h): reduce score

Complete the function below:

In [ ]:
# ═══════════════════════════════════════════════════════════
# 9. TASK 6.1: BIKEABILITY SCORE
# ═══════════════════════════════════════════════════════════

def calculate_bikeability_score(row):
    """
    Calculate a bikeability score (0-1) based on weather conditions.
    
    Higher score = better conditions for biking
    
    Parameters:
    -----------
    row : pandas Series
        Row with weather data
    
    Returns:
    --------
    float : Score between 0 and 1
    """
    score = 1.0
    
    # TODO: Adjust score based on temperature
    temp = row['temperature_2m']
    # Hint: if temp < 5 or temp > 30: score *= 0.3
    # Hint: elif temp < 15 or temp > 25: score *= 0.7
    
    # TODO: Adjust score based on precipitation
    precip = row['precipitation']
    # Hint: if precip > 5: score *= 0.2
    # Hint: elif precip > 0: score *= 0.6
    
    # TODO: Adjust score based on wind speed
    wind = row['windspeed_10m']
    # Hint: if wind > 25: score *= 0.5
    
    return score


# Apply the function (will work after you implement it)
df_weather['bikeability_score'] = df_weather.apply(calculate_bikeability_score, axis=1)

print("📊 Bikeability Score Statistics:")
print(f"Mean: {df_weather['bikeability_score'].mean():.2f}")
print(f"Min: {df_weather['bikeability_score'].min():.2f}")
print(f"Max: {df_weather['bikeability_score'].max():.2f}")

# Plot bikeability over time
plt.figure(figsize=(14, 5))
plt.plot(df_weather['timestamp'], df_weather['bikeability_score'], 
         color='darkgreen', linewidth=2, alpha=0.8)
plt.axhline(df_weather['bikeability_score'].mean(), color='red', 
            linestyle='--', label=f"Mean: {df_weather['bikeability_score'].mean():.2f}")
plt.fill_between(df_weather['timestamp'], 0, df_weather['bikeability_score'], 
                 alpha=0.3, color='green')
plt.xlabel('Date')
plt.ylabel('Bikeability Score (0-1)')
plt.title('🚴 Bikeability Score Over Time (Based on Weather)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 🧠 Task 6.2: Weather Pattern Analysis

**Your Task**: Analyze weather patterns and answer these questions:

1. What was the coldest hour? Display temperature, time, and other conditions
2. What was the rainiest hour? Show precipitation amount and conditions
3. Calculate the percentage of time with "good biking weather" (score > 0.7)
4. Find the longest stretch of consecutive hours with no rain

**Hints:**
- Use `.idxmin()` and `.idxmax()` to find extreme values
- Use `.loc[index]` to get the full row
- Boolean filtering: `df[df['column'] > threshold]`
- For consecutive hours: look for groups where precipitation == 0

In [ ]:
# ═══════════════════════════════════════════════════════════
# TASK 6.2: WEATHER PATTERN ANALYSIS
# ═══════════════════════════════════════════════════════════

# TODO: Analyze weather patterns
# Write your code here



### 🧠 Task 6.3: Create Weather Comparison Visualization

**Your Task**: Create a 2x2 subplot figure comparing weather variables:

**Panel 1 (Top-Left):** Temperature distribution histogram
- 30 bins, color='coral'
- Add mean line in red
- Label axes and add title

**Panel 2 (Top-Right):** Precipitation over time
- Bar chart with timestamp on x-axis
- Color='steelblue', alpha=0.7
- Show total rainfall in title

**Panel 3 (Bottom-Left):** Wind speed vs Temperature scatter
- Color points by bikeability_score (cmap='RdYlGn')
- Add colorbar
- Show correlation in title

**Panel 4 (Bottom-Right):** Bikeability score distribution
- Histogram with 20 bins
- Color='green', add mean line
- Label percentage of "good" hours (score > 0.7)

**Hints:**
- `fig, axes = plt.subplots(2, 2, figsize=(16, 12))`
- Access subplots: `axes[0, 0]`, `axes[0, 1]`, etc.
- For scatter: `axes[row, col].scatter(x, y, c=colors, cmap='RdYlGn')`
- Calculate correlation: `df['col1'].corr(df['col2'])`

In [ ]:
# ═══════════════════════════════════════════════════════════
# TASK 6.3: CREATE WEATHER COMPARISON VISUALIZATION
# ═══════════════════════════════════════════════════════════

# TODO: Create 2x2 subplot figure
# Write your code here



### 🧠 Task 6.4: Custom Analysis (Your Choice!)

**Your Task**: Create ONE additional analysis or visualization that answers an interesting question about weather patterns.

**Ideas:**
- How do weather patterns differ by time of day?
- Is there a relationship between humidity and precipitation?
- What's the "typical" weather profile (median values by hour)?
- Create a weather "comfort index" combining multiple factors
- Analyze trends over the 30-day period

**Requirements:**
- Include a clear title explaining your analysis
- Add appropriate labels and formatting
- Write 2-3 sentences interpreting your findings
- Show creativity in your approach!

In [ ]:
# ═══════════════════════════════════════════════════════════
# TASK 6.4: CUSTOM ANALYSIS
# ═══════════════════════════════════════════════════════════

# TODO: Your custom weather analysis here
# Write your code here



---## 🔥 Part 6a: Optional Advanced Challenges**For advanced learners**: These challenges push beyond the basics. Try at least ONE!### Challenge 7.1: Explore KNMI Weather API 🇳🇱**Task:** Fetch weather data from KNMI (Royal Netherlands Meteorological Institute) and compare with Open-Meteo data for Amsterdam.**Why this matters:** In real projects, you often need to compare data from multiple sources for validation, or choose between different providers. KNMI is the official Dutch weather service and provides authoritative data for the Netherlands.**Requirements:**- Research KNMI's open data API documentation (no API key required)- Fetch historical weather data for Amsterdam from KNMI- Parse KNMI's data format (different from Open-Meteo!)- Compare temperature and precipitation with your Open-Meteo data- Identify any significant differences and hypothesize why they might exist**Starting points to explore:**- KNMI Open Data Portal: https://www.daggegevens.knmi.nl/klimatologie/daggegevens- KNMI API documentation: Look for hourly data endpoints- Consider: How does KNMI structure their response? JSON? CSV? XML?- Consider: What weather stations are near Amsterdam? How do you specify location?**Hints:**- KNMI uses different variable names (e.g., "T" for temperature)- Data might be in different units (tenths of degrees Celsius)- Station-based data (not coordinates) - find Amsterdam/Schiphol station code- You'll need to adapt your data parsing code significantly!**Success criteria:**- Successfully fetch KNMI data without copying solution code- Create a comparison plot showing both data sources- Document 2-3 key differences you discovered in API structure- Brief reflection: Which API was easier to use and why?**Pedagogical goal:** This challenge builds your ability to read documentation and adapt to new APIs independently - a critical real-world skill!### Challenge 7.2: Weather Forecast Integration 📊**Task:** Fetch 7-day weather forecast and visualize predicted bikeability.**Requirements:**- Use Open-Meteo forecast API: `https://api.open-meteo.com/v1/forecast`- Calculate predicted bikeability scores- Create visualization showing next 7 days- Identify best days for bike rides**Hint:** Forecast API uses similar parameters but no `start_date/end_date`### Challenge 7.3: Weather Data Caching System 💾**Task:** Implement a caching system to avoid redundant API calls.**Requirements:**- Cache responses in JSON files with timestamp- Check if cached data is recent (< 1 hour old) before making new API call- Create function `fetch_weather_with_cache(lat, lon, start, end, cache_dir)`- Include cache hit/miss statistics### Challenge 7.4: Extreme Weather Alerting 🚨**Task:** Create a function that identifies and alerts on extreme weather conditions.**Requirements:**- Define thresholds: temp < 0°C or > 35°C, rain > 10mm/hr, wind > 40 km/h- Function returns list of extreme weather events with timestamps- Create visualization highlighting extreme periods- Generate text summary of alerts**Example output:** "⚠️ Extreme cold: 3 hours below 0°C on 2026-01-05"

In [ ]:
# ═══════════════════════════════════════════════════════════
# PART 6A: OPTIONAL ADVANCED CHALLENGES
# ═══════════════════════════════════════════════════════════

# TODO: Implement one or more advanced challenges here
# Write your code here



---

## ✅ Part 7: Data Validation

Validate weather data quality before saving.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 10. DATA VALIDATION
# ═══════════════════════════════════════════════════════════

def validate_weather_data(df):
    """
    Validate weather data quality.
    
    Returns:
    --------
    bool : True if all validations pass
    """
    validations = []
    
    # Check 1: Required columns exist
    required_cols = ['timestamp', 'temperature_2m', 'precipitation']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f"❌ Missing columns: {missing_cols}")
        validations.append(False)
    else:
        print(f"✅ All required columns present")
        validations.append(True)
    
    # Check 2: No missing timestamps
    if df['timestamp'].isna().any():
        print(f"❌ Found missing timestamps")
        validations.append(False)
    else:
        print(f"✅ All timestamps present")
        validations.append(True)
    
    # Check 3: Temperature in reasonable range
    if (df['temperature_2m'] < -50).any() or (df['temperature_2m'] > 50).any():
        print(f"❌ Temperature out of reasonable range (-50 to 50°C)")
        validations.append(False)
    else:
        print(f"✅ Temperature values reasonable")
        validations.append(True)
    
    # Check 4: Precipitation is non-negative
    if (df['precipitation'] < 0).any():
        print(f"❌ Found negative precipitation values")
        validations.append(False)
    else:
        print(f"✅ Precipitation values valid")
        validations.append(True)
    
    # Check 5: Wind speed is non-negative
    if (df['windspeed_10m'] < 0).any():
        print(f"❌ Found negative wind speed values")
        validations.append(False)
    else:
        print(f"✅ Wind speed values valid")
        validations.append(True)
    
    # Check 6: Humidity in valid range (0-100%)
    if (df['relativehumidity_2m'] < 0).any() or (df['relativehumidity_2m'] > 100).any():
        print(f"❌ Humidity out of valid range (0-100%)")
        validations.append(False)
    else:
        print(f"✅ Humidity values valid")
        validations.append(True)
    
    # Check 7: Continuous time series (no gaps)
    time_diffs = df['timestamp'].diff().dropna()
    expected_diff = pd.Timedelta(hours=1)
    if (time_diffs != expected_diff).any():
        gaps = (time_diffs != expected_diff).sum()
        print(f"⚠️ Warning: Found {gaps} time gaps in hourly data")
        validations.append(True)  # Warning, not error
    else:
        print(f"✅ Continuous hourly time series")
        validations.append(True)
    
    return all(validations)


# Run validation
print("🔍 Validating weather data quality...")
print("=" * 60)
is_valid = validate_weather_data(df_weather)
print("=" * 60)

if is_valid:
    print("✅ All validation checks passed!")
else:
    print("❌ Some validation checks failed - review data before saving")

---

## 💾 Part 8: Save Weather Data

Save the processed weather data with documentation.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 11. SAVE WEATHER DATA
# ═══════════════════════════════════════════════════════════

# Save as CSV
print("💾 Saving weather data...")
df_weather.to_csv(OUTPUT_FILE, index=False)
print(f"✅ Saved CSV: {OUTPUT_FILE}")
print(f"📦 File size: {OUTPUT_FILE.stat().st_size / 1024:.1f} KB")

# Create metadata file
metadata = {
    'source': 'Open-Meteo Archive API',
    'api_url': BASE_URL,
    'location': {
        'city': 'Amsterdam',
        'country': 'Netherlands',
        'latitude': weather_data['latitude'],
        'longitude': weather_data['longitude'],
        'elevation_m': weather_data['elevation']
    },
    'fetch_timestamp': datetime.now().isoformat(),
    'data_period': {
        'start': START_DATE.isoformat(),
        'end': END_DATE.isoformat(),
        'num_records': len(df_weather)
    },
    'weather_variables': WEATHER_VARS,
    'timezone': weather_data['timezone'],
    'summary': {
        'avg_temperature_c': float(df_weather['temperature_2m'].mean()),
        'total_precipitation_mm': float(df_weather['precipitation'].sum()),
        'avg_windspeed_kmh': float(df_weather['windspeed_10m'].mean()),
        'avg_bikeability_score': float(df_weather['bikeability_score'].mean())
    }
}

metadata_file = OUTPUT_FILE.with_suffix('.metadata.json')
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✅ Saved metadata: {metadata_file}")

print("\n" + "=" * 60)
print("📊 Data Acquisition Summary:")
print("=" * 60)
print(f"Records fetched: {len(df_weather)}")
print(f"Date range: {df_weather['timestamp'].min()} to {df_weather['timestamp'].max()}")
print(f"Weather variables: {len(WEATHER_VARS)}")
print(f"Files saved:")
print(f"  • {OUTPUT_FILE.name}")
print(f"  • {metadata_file.name}")

---

## 📝 Part 9: Summary

### What You've Learned ✅

In this notebook, you:
1. ✅ Connected to the Open-Meteo weather API
2. ✅ Fetched historical weather data for Amsterdam
3. ✅ Parsed and cleaned weather variables
4. ✅ Created derived features (bikeability score)
5. ✅ Validated weather data quality
6. ✅ Visualized weather patterns over time
7. ✅ Saved processed weather data with documentation

### Key Takeaways 💡

1. **Weather APIs** provide rich historical and forecast data
2. **Timezone matters** - always specify correct timezone for timestamps
3. **Domain knowledge** helps create meaningful features (bikeability score)
4. **Data validation** catches unusual values and gaps
5. **Hourly data** provides detailed patterns but requires more storage

### Data Files Created 📁

- `data/raw/weather_amsterdam_{timestamp}.csv` - Processed weather data
- `data/raw/weather_amsterdam_{timestamp}.metadata.json` - Data documentation

### Next Steps 🚀

Now that you have both bike and weather data, proceed to:
- **M2_03_data_storage.ipynb** - Learn storage best practices
- **M2_04_merge_datasets.ipynb** - Combine bike and weather data

### 🧠 Reflection Questions

1. **How might weather forecasts** be used to predict future bike availability?
2. **What other weather variables** could be relevant for bike usage?
3. **How would you handle** very long time periods (e.g., 5 years of data)?
4. **What seasonal patterns** do you expect to see in bike usage based on weather?

**Write your reflections below** ⬇️

### My Reflections

[Your thoughts here]

---

## 📚 References

- [Open-Meteo API Documentation](https://open-meteo.com/en/docs)
- [Weather Variable Definitions](https://open-meteo.com/en/docs/historical-weather-api)
- [Pandas Time Series](https://pandas.pydata.org/docs/user_guide/timeseries.html)
- [Datetime Handling in Python](https://docs.python.org/3/library/datetime.html)

---

**🎉 Congratulations!** You've successfully completed M2_02 - Weather Data Acquisition!